# Problema blocuri

Fie M spații și N blocuri cu greutăți diferite într-o configurație oarecare. Putem muta blocul din vârful unei stive pe orice altă stivă, indiferent de greutatea blocurilor, dar costul mutării va fi egal cu greutatea blocului mutat. Scopul jocului este să mutăm toate piesele într-o configurație dată.

![Image](https://drive.google.com/uc?export=view&id=1a0iavgQuWHQys3C2v_JSTHJE3A3KqsC4)


## Exercițiul 1

Rezolvați problema blocurilor cu bfs/dfs, folosind clasele Nod și Graf discutate până acum

In [2]:
class Nod:
    def __init__(self, stive, parinte=None, cost=0, cost_total=0):
        self.stive = [list(s) for s in stive]
        self.parinte = parinte
        self.cost = cost
        self.cost_total = cost_total

    def __eq__(self, other):
        return self.stive == other.stive

    def __str__(self):
        linii = []
        for i, stiva in enumerate(self.stive):
            continut = ' '.join(str(b) for b in stiva) if stiva else 'stiva goala'
            linii.append(f"stiva {i}: [{continut}]  <- varf")
        return "stare curenta:\n" + "\n".join(linii)

    def __repr__(self):
        return f"nod({self.stive})"

    def drum_radacina(self):
        noduri = []
        temp = self
        while temp is not None:
            noduri.append(temp)
            temp = temp.parinte
        noduri.reverse()
        return noduri

    def vizitat(self, nod):
        for n_vechi in self.drum_radacina():
            if nod == n_vechi:
                return True
        return False


class Graf:
    def __init__(self, nod_start, stare_scop):
        self.nod_start = nod_start
        self.stare_scop = [list(s) for s in stare_scop]

    def scop(self, nod):
        return nod.stive == self.stare_scop

    def succesori(self, nod):
        lista_succesori = []
        m = len(nod.stive)

        for i in range(m):
            if not nod.stive[i]:
                continue
            bloc = nod.stive[i][-1]
            cost_mutare = bloc

            for j in range(m):
                if i == j:
                    continue
                if nod.stive[j] and bloc > nod.stive[j][-1]:
                    continue

                noi_stive = [list(s) for s in nod.stive]
                noi_stive[i].pop()
                noi_stive[j].append(bloc)

                nod_nou = Nod(
                    stive=noi_stive,
                    parinte=nod,
                    cost=cost_mutare,
                    cost_total=nod.cost_total + cost_mutare
                )
                if not nod.vizitat(nod_nou):
                    lista_succesori.append(nod_nou)

        return lista_succesori

def print_drum(nod):
    print(f"Cost: {nod.cost_total}")

def bfs(graf, n_solutii=1):
    coada = [graf.nod_start]
    solutii = 0

    while coada and solutii < n_solutii:
        curent = coada.pop(0)
        if graf.scop(curent):
            solutii += 1
            print(f"\n=== Solutia {solutii} (BFS) ===")
            print_drum(curent)
            if solutii == n_solutii:
                break
        for suc in graf.succesori(curent):
            coada.append(suc)

    if solutii == 0:
        print("Nu s-a gasit nicio solutie.")


## Exercițiul 2

*Pregătirea pentru algoritmul A* *

Putem alege mai eficient următorul pas luând în calcul nu doar costul drumului, ci și estimarea costului de la un nod până la final (*euristica*).

Pentru o stare a jocului, dați exemplu de câte o euristică:
- neadmisibilă
- admisibilă neconsistentă
- admisibilă consistentă


Spunem că o euristică este **admisibilă** dacă valoarea estimării este mai mică sau egală decât costul celui mai scurt drum de la nodul respectiv la oricare dintre nodurile scop.

Spunem că o euristică \hat{h} este **consistentă** dacă este monotonă:  $\hat{h}(n_i) \leq cost(n_i \rightarrow n_j) + \hat{h}(n_j)$, pentru orice $\{n_i, n_j\}$ aparținând grafului de stări, unde $n_j$ este succesor direct al lui $n_i$


Fiecare euristică va avea propria ei funcție, care va fi setată când rulăm prima dată programul.

In [ ]:
'''
 euristica: estimarea optimista a costului pana la final (i guess)
- aici: adun costul blocurilor care nu se afla pe pozitia finala (aka cel putin cate o mutare pt fiecare bloc)
- admisibil: estimarea e intotdeauna <= decat costul real
aici e admisibila ptc trebuie sa mutam macar o data fiecare bloc care nu e unde trebuie, hence costul estimat
+ restrictiile impuse care probabil ca ar insemna 0 sau alte mutari in plus
- alt exemplu de euristica admisibila in cazul asta: numar cate blocuri nu sunt la locul lor
- alta: 0 peste tot :)) (e dijkstra)
- alta: nr blocuri * costul minim
- alta: 0 daca sunt la final, 1 otherwise
'''
'''
consistenta: euristica nod 1 <= cost (n1n2) + euristica nod 2 (evident ig)
- uneori suntem nevoiti sa ajungem intr o stare mai proasta dintr o stare mai buna, deci nu e intotdeasuna consistenta
- euristica buna: admisibila si consistenta
- exemplu de euristica admisibila neconsistenta pentru situatia cu blocuri:

'''